# NOPP-Aleutians Wirewalker RBR Concerto CTD — processing

Trimmed view of the processing products (L1 → L2 → L3 → L3i), ending at the L3 interpolated sections. Derived from `wirewalker_ctd_plots.ipynb`.


In [ ]:
import os
import time as _time
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import gsw
import json
from pathlib import Path

# Locate the repo (the folder containing config.json) by walking up from the working dir,
# then read paths from the config -- no hardcoded machine paths.
def _find_repo(start):
    for d in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (d / 'config.json').exists():
            return d
    raise FileNotFoundError(f'config.json not found at/above {start}')

REPO = _find_repo(Path.cwd())
CFG  = json.load(open(REPO / 'config.json'))
OUT  = Path(os.environ.get('WW_OUTPUT_DIR', CFG['output_dir'])).expanduser()  # data products
BASE = CFG['basename']
FIGS = REPO / 'figs'; FIGS.mkdir(exist_ok=True)                              # figures stay in the repo

L1_FILE  = OUT / 'L1' / f'{BASE}_L1_converted.nc'
L2_FILE  = OUT / 'L2' / f'{BASE}_L2_upcast_grid0.5m.nc'
L3_FILE  = OUT / 'L3' / f'{BASE}_L3_grid1m_30min.nc'
L3I_FILE = OUT / 'L3' / f'{BASE}_L3_grid1m_30min_interp.nc'

# close any handles left open by a previous run in this kernel
for _v in ('L1', 'L2', 'L2t', 'L3', 'L3i'):
    if _v in globals():
        try:
            globals()[_v].close()
        except Exception:
            pass

L1  = xr.open_dataset(L1_FILE)
L2  = xr.open_dataset(L2_FILE)
L3  = xr.open_dataset(L3_FILE)            # regular 1 m x 30 min grid (empty bins = NaN)
L3i = xr.open_dataset(L3I_FILE)           # same grid, single empty bins interpolated
# L2 carries cast time as a coord on the 'cast' dim; promote it to a usable axis
L2t = L2.assign_coords(time=L2['time']).swap_dims({'cast': 'time'}).sortby('time')

# --- freshness check: confirm we loaded the files we think we did ---
print('repo:', REPO, '| data:', OUT)
for tag, f in (('L1', L1_FILE), ('L2', L2_FILE), ('L3', L3_FILE), ('L3i', L3I_FILE)):
    st = f.stat()
    print(f'{tag:4s} {st.st_size/1e6:6.0f} MB  modified {_time.ctime(st.st_mtime)}')
print('L1 vars:', list(L1.data_vars))
print('L2 vars:', list(L2.data_vars), '| upcasts:', L2.sizes['cast'])
print('L3 vars:', list(L3.data_vars), '| grid:', dict(L3.sizes))

## 1. Vehicle depth vs time (L1) &mdash; cast-flag check
Full-deployment coverage as **one calendar month per panel, 4 panels per page** (writes a multi-page PDF). Colored by `cast_direction`: **upcast = red, downcast = blue**. Decimated per panel so it renders fast; use the zoom cell below for a full 2 Hz look at the up/down sawtooth.

In [ ]:
# Two-week depth-vs-time panels (4 rows per page) over the whole deployment.
# upcast = red, downcast = blue; decimated per panel for plotting speed.
# One PNG written per page.
import pandas as pd
from matplotlib.lines import Line2D

PER_PAGE = 4
TARGET   = 120000                 # ~points plotted per 2-week panel
SAVE_PNG = True                   # write L1_depth_time_2wk_pageN.png per page

t0 = pd.Timestamp(L1.time.values[0]); t1 = pd.Timestamp(L1.time.values[-1])
edges  = pd.date_range(t0.floor('D'), t1 + pd.Timedelta('14D'), freq='14D')
panels = list(zip(edges[:-1], edges[1:]))
pages  = [panels[i:i + PER_PAGE] for i in range(0, len(panels), PER_PAGE)]

legend = [Line2D([], [], marker='o', ls='', color='tab:red',  label='upcast'),
          Line2D([], [], marker='o', ls='', color='tab:blue', label='downcast')]

for pg, page in enumerate(pages, 1):
    fig, axes = plt.subplots(PER_PAGE, 1, figsize=(12, 11), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (m0, m1) in zip(axes, page):
        sub  = L1.sel(time=slice(m0, m1))
        n    = sub.sizes['time']; step = max(1, n // TARGET)
        tt   = sub['time'].values[::step]
        dd   = sub['depth'].values[::step]
        dr   = sub['cast_direction'].values[::step]
        up, dn = dr == 1, dr == 0
        ax.scatter(tt[dn], dd[dn], s=2, c='tab:blue', edgecolors='none', rasterized=True)
        ax.scatter(tt[up], dd[up], s=2, c='tab:red',  edgecolors='none', rasterized=True)
        ax.set_xlim(m0, m1); ax.set_ylabel('Depth (m)')
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.set_title(f'{m0.date()} – {m1.date()}   (n={n:,}, every {step}th sample)',
                     fontsize=9, loc='left')
        ax.grid(alpha=0.3)
    axes[0].invert_yaxis()                # surface at top, depth increasing downward
    for ax in axes[len(page):]:           # blank unused panels on the last page
        ax.axis('off')
    axes[len(page) - 1].set_xlabel('Date')
    fig.legend(handles=legend, loc='upper right', ncol=2, markerscale=1.5)
    fig.suptitle(f'L1 vehicle depth vs time — page {pg}/{len(pages)}', y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.99))
    if SAVE_PNG:
        out = FIGS / f'L1_depth_time_2wk_page{pg}.png'
        fig.savefig(out, dpi=130); print('wrote', out)

In [ ]:
# Full-resolution zoom: pick a short window to verify the up/down sawtooth.
TWIN = ('2025-09-01 00:00', '2025-09-01 06:00')   # (start, end) — a 6-hour window

sub  = L1.sel(time=slice(*TWIN))
tt   = sub['time'].values
dd   = sub['depth'].values
dr   = sub['cast_direction'].values
up, dn = dr == 1, dr == 0

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(tt[dn], dd[dn], s=4, c='tab:blue', edgecolors='none', label='downcast')
ax.scatter(tt[up], dd[up], s=4, c='tab:red',  edgecolors='none', label='upcast')
ax.invert_yaxis(); ax.set_ylabel('Depth (m)'); ax.set_xlabel('Time')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title(f'L1 depth vs time — {TWIN[0]} to {TWIN[1]} (full 2 Hz, n={tt.size:,})')
ax.legend(markerscale=3, loc='upper right'); ax.grid(alpha=0.3)
fig.tight_layout()

## 2. Deployment overview &mdash; time vs depth sections (L2 upcasts)

In [ ]:
fields = [('conservative_temperature', 'CT (\u00b0C)', 'thermal'),
          ('practical_salinity',       'Salinity (PSU)', 'haline'),
          ('sigma0',                   '\u03c3\u2080 (kg m\u207b\u00b3)', 'dense')]
try:
    import cmocean
    cmaps = {'thermal': cmocean.cm.thermal, 'haline': cmocean.cm.haline, 'dense': cmocean.cm.dense}
except ImportError:
    cmaps = {'thermal': 'inferno', 'haline': 'viridis', 'dense': 'cividis'}

fig, axes = plt.subplots(len(fields), 1, figsize=(13, 9), sharex=True)
for ax, (var, label, cm) in zip(axes, fields):
    da = L2t[var]
    p = ax.pcolormesh(L2t['time'].values, L2t['depth'].values, da.values,
                      cmap=cmaps[cm], shading='nearest')
    ax.invert_yaxis()
    ax.set_ylabel('Depth (m)')
    fig.colorbar(p, ax=ax, label=label, pad=0.01)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
axes[0].set_title('Wirewalker upcasts \u2014 ' + str(L2.attrs.get('mooring', '')))
fig.tight_layout()

## 3. Individual profiles (T, S, sigma vs depth)

In [ ]:
# pick a handful of upcasts spread across the deployment
ncast = L2.sizes['cast']
idx = np.linspace(0, ncast - 1, min(6, ncast)).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(11, 6), sharey=True)
for j in idx:
    c = L2.isel(cast=j)
    lab = str(np.datetime64(c['time'].values, 'h'))
    ax[0].plot(c['conservative_temperature'], L2['depth'], label=lab)
    ax[1].plot(c['practical_salinity'], L2['depth'])
    ax[2].plot(c['sigma0'], L2['depth'])
for a, t in zip(ax, ['CT (\u00b0C)', 'Salinity (PSU)', '\u03c3\u2080 (kg m\u207b\u00b3)']):
    a.set_xlabel(t); a.grid(alpha=0.3)
ax[0].invert_yaxis(); ax[0].set_ylabel('Depth (m)')
ax[0].legend(fontsize=7, title='upcast')
fig.tight_layout()

## 4. T-S diagram with potential-density contours

In [ ]:
# T-S diagram. L2 has ~14M bins, so subsample to keep the scatter fast.
MAXPTS = 200_000

SP  = L2['practical_salinity'].values.ravel()
CT  = L2['conservative_temperature'].values.ravel()
dep = np.broadcast_to(L2['depth'].values[:, None], L2['practical_salinity'].shape).ravel()
good = np.isfinite(SP) & np.isfinite(CT)
SP, CT, dep = SP[good], CT[good], dep[good]
step = max(1, SP.size // MAXPTS)          # decimate to ~MAXPTS points
SPp, CTp, depp = SP[::step], CT[::step], dep[::step]

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(SPp, CTp, c=depp, s=2, cmap='viridis_r', edgecolors='none', rasterized=True)
fig.colorbar(sc, label='Depth (m)')
# potential-density contours (coarse grid over the data range)
sg = np.linspace(SP.min(), SP.max(), 100)
tg = np.linspace(CT.min(), CT.max(), 100)
Sg, Tg = np.meshgrid(sg, tg)
SAg = gsw.SA_from_SP(Sg, 0, L2.attrs['geospatial_lon'], L2.attrs['geospatial_lat'])
cs = ax.contour(Sg, Tg, gsw.sigma0(SAg, Tg), colors='grey', linewidths=0.7)
ax.clabel(cs, fmt='%.1f', fontsize=7)
ax.set_xlabel('Practical salinity (PSU)'); ax.set_ylabel('CT (°C)')
ax.set_title(f'T–S (L2 upcast bins, {SPp.size:,} of {SP.size:,} pts)')
fig.tight_layout()

## 4b. Seasonal T–S/depth
Same T–S/depth view split into three seasons of the deployment — **summer/early fall (Jul–Oct 2025)**, **late fall/winter (Nov 2025–Feb 2026)**, **spring/early summer (Mar–May 2026)** — with shared axes, σ₀ contours and depth colorbar. Writes `TS_seasonal.png`.

In [ ]:
# Seasonal T-S/depth: three panels over the deployment.
MAXPTS = 120_000
seasons = [("Summer / early fall\n(Jul–Oct 2025)",     "2025-07-01", "2025-11-01"),
           ("Late fall / winter\n(Nov 2025–Feb 2026)", "2025-11-01", "2026-03-01"),
           ("Spring / early summer\n(Mar–May 2026)",   "2026-03-01", "2026-06-01")]

ctime  = L2['time'].values
SPall  = L2['practical_salinity'].values
CTall  = L2['conservative_temperature'].values
depcol = np.broadcast_to(L2['depth'].values[:, None], SPall.shape)
g = np.isfinite(SPall) & np.isfinite(CTall)
sx = (np.nanpercentile(SPall[g], 0.1), np.nanpercentile(SPall[g], 99.9))
tx = (np.nanmin(CTall[g]), np.nanmax(CTall[g]))

# shared potential-density grid over the global data range
sg = np.linspace(*sx, 120); tg = np.linspace(*tx, 120); Sg, Tg = np.meshgrid(sg, tg)
sig = gsw.sigma0(gsw.SA_from_SP(Sg, 0, L2.attrs['geospatial_lon'], L2.attrs['geospatial_lat']), Tg)

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharex=True, sharey=True)
for ax, (lab, t0, t1) in zip(axes, seasons):
    m = (ctime >= np.datetime64(t0)) & (ctime < np.datetime64(t1))
    SP = SPall[:, m].ravel(); CT = CTall[:, m].ravel(); dp = depcol[:, m].ravel()
    gd = np.isfinite(SP) & np.isfinite(CT); SP, CT, dp = SP[gd], CT[gd], dp[gd]
    st = max(1, SP.size // MAXPTS)
    cs = ax.contour(Sg, Tg, sig, colors='grey', linewidths=0.6); ax.clabel(cs, fmt='%.1f', fontsize=6)
    sc = ax.scatter(SP[::st], CT[::st], c=dp[::st], s=2, cmap='viridis_r',
                    vmin=0, vmax=500, edgecolors='none', rasterized=True)
    ax.set_title(f'{lab}\n{int(m.sum())} upcasts, {SP[::st].size:,} pts', fontsize=9)
    ax.set_xlabel('Practical salinity (PSU)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('CT (°C)'); axes[0].set_xlim(sx); axes[0].set_ylim(tx)
fig.colorbar(sc, ax=axes, label='Depth (m)', pad=0.01, fraction=0.025)
fig.suptitle('Seasonal T–S (L2 upcasts), colored by depth', y=1.02)
fig.savefig(FIGS / 'TS_seasonal.png', dpi=130, bbox_inches='tight'); print('wrote', FIGS / 'TS_seasonal.png')

## 5. Using the L1 cast flags &mdash; raw up vs down for one profile
Picks a single Ruskin profile from the full-resolution L1 record using the `profile_number` / `cast_direction` flags. L1 holds only raw channels (conductivity, temperature, pressure, depth), so practical salinity is computed here on the fly with `gsw`.

In [ ]:
prof = int(np.median(np.unique(L1['profile_number'].values)))
sel = L1.where(L1['profile_number'] == prof, drop=True)

ATM = L2.attrs.get('atmospheric_pressure_dbar', 10.1325)
sea_p = sel['pressure'] - ATM
SP = gsw.SP_from_C(sel['conductivity'], sel['temperature'], sea_p)   # practical salinity

fig, ax = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
for d, name, color in [(0, 'down', 'tab:blue'), (1, 'up', 'tab:red')]:
    m = sel['cast_direction'] == d
    ax[0].plot(sel['temperature'].where(m, drop=True),
               sel['depth'].where(m, drop=True), '.', ms=2, color=color, label=name)
    ax[1].plot(SP.where(m, drop=True),
               sel['depth'].where(m, drop=True), '.', ms=2, color=color, label=name)
ax[0].set_xlabel('Temperature (°C)'); ax[1].set_xlabel('Practical salinity (PSU)')
ax[0].set_ylabel('Depth (m)'); ax[0].invert_yaxis()
ax[0].legend(); [a.grid(alpha=0.3) for a in ax]
fig.suptitle(f'L1 profile #{prof}: 2 Hz raw, down vs up cast')
fig.tight_layout()

## 6. Mean profile and stratification

In [ ]:
mean_ct = L2['conservative_temperature'].mean('cast')
mean_sp = L2['practical_salinity'].mean('cast')
mean_sig = L2['sigma0'].mean('cast')
fig, ax = plt.subplots(1, 3, figsize=(11, 6), sharey=True)
ax[0].plot(mean_ct, L2['depth']); ax[0].set_xlabel('mean CT (\u00b0C)')
ax[1].plot(mean_sp, L2['depth']); ax[1].set_xlabel('mean Salinity')
ax[2].plot(mean_sig, L2['depth']); ax[2].set_xlabel('mean \u03c3\u2080')
ax[0].invert_yaxis(); ax[0].set_ylabel('Depth (m)'); [a.grid(alpha=0.3) for a in ax]
fig.suptitle('Deployment-mean upcast profile'); fig.tight_layout()

## 7. L3 regular grid — deployment sections
Depth–time sections (Hovmöller) from the **L3** product (1 m × 30 min regular grid, Nyquist 1 cph). Empty 30-min bins (~8%, from the cadence/bin phase slip) show as gaps. Writes `L3_sections.png`.

In [ ]:
# L3 regular grid (1 m x 30 min): full-deployment depth-time sections.
fields = [('conservative_temperature', 'CT (°C)', 'thermal'),
          ('practical_salinity',       'Salinity (PSU)', 'haline'),
          ('sigma0',                   'σ₀ (kg m⁻³)', 'dense')]
try:
    import cmocean
    cmaps = {'thermal': cmocean.cm.thermal, 'haline': cmocean.cm.haline, 'dense': cmocean.cm.dense}
except ImportError:
    cmaps = {'thermal': 'inferno', 'haline': 'viridis', 'dense': 'cividis'}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, (var, label, cm) in zip(axes, fields):
    p = ax.pcolormesh(L3['time'].values, L3['depth'].values, L3[var].values,
                      cmap=cmaps[cm], shading='nearest', rasterized=True)
    ax.invert_yaxis(); ax.set_ylabel('Depth (m)')
    fig.colorbar(p, ax=ax, label=label, pad=0.01)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].set_title('L3 regular grid (1 m × 30 min) — NOPP-Aleutians Wirewalker')
fig.tight_layout()
fig.savefig(FIGS / 'L3_sections.png', dpi=130); print('wrote', FIGS / 'L3_sections.png')

## 7b. L3 interpolated — deployment sections
Same sections from the **gap-filled L3 companion** (`L3i`): single empty 30-min bins linearly interpolated (≈8% → ~1% gaps), so the phase-slip stripes are gone. Real multi-hour gaps remain. Writes `L3_sections_interp.png`.

In [ ]:
# L3 interpolated companion: same sections, single empty bins gap-filled (cleaner).
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, (var, label, cm) in zip(axes, fields):
    p = ax.pcolormesh(L3i['time'].values, L3i['depth'].values, L3i[var].values,
                      cmap=cmaps[cm], shading='nearest', rasterized=True)
    ax.invert_yaxis(); ax.set_ylabel('Depth (m)')
    fig.colorbar(p, ax=ax, label=label, pad=0.01)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].set_title('L3 interpolated (1 m × 30 min, single gaps filled) — NOPP-Aleutians Wirewalker')
fig.tight_layout()
fig.savefig(FIGS / 'L3_sections_interp.png', dpi=130); print('wrote', FIGS / 'L3_sections_interp.png')